In [1]:
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

In [2]:
# =============== 领域规范（与 rag_2 完全一致） ===============
NER_TYPES = {
    "CROP": ("作物", "Crop"),
    "VAR": ("品种", "Variety/Cultivar"),
    "TRT": ("性状", "Trait"),
    "GST": ("生育时期", "Growth Stage"),
    "GENE": ("基因", "Gene"),
    "QTL": ("数量性状位点", "QTL"),
    "MRK": ("分子标记", "Molecular Marker"),
    "CHR": ("染色体", "Chromosome"),
    "BM": ("育种方法", "Breeding Method"),
    "CROSS": ("亲本/杂交组合", "Parent/Cross"),
    "ABS": ("非生物胁迫", "Abiotic Stress"),
    "BIS": ("生物胁迫", "Biotic Stress")
}

RE_RELATIONS = {
    "CON": {"name": "包含", "desc": "品种属于某作物", "head": ["VAR"], "tail": ["CROP"]},
    "USE": {"name": "采用", "desc": "品种采用某种育种方法", "head": ["VAR"], "tail": ["BM"]},
    "HAS": {"name": "具有", "desc": "品种具备或关注某性状", "head": ["VAR"], "tail": ["TRT"]},
    "AFF": {"name": "影响", "desc": "非生物胁迫、基因、分子标记或 QTL 影响性状", "head": ["ABS", "GENE", "MRK", "QTL"], "tail": ["TRT"]},
    "OCI": {"name": "发生于", "desc": "性状或胁迫发生于某生育时期", "head": ["TRT", "ABS", "BIS"], "tail": ["GST"]},
    "LOI": {"name": "定位于", "desc": "分子标记、QTL 或基因定位于染色体或区间", "head": ["MRK", "QTL", "GENE"], "tail": ["CHR", "QTL"]}
}

VALID_ENTITY_LABELS = set(NER_TYPES.keys())
VALID_RELATION_LABELS = set(RE_RELATIONS.keys())


class ZeroShotNERREPredictor:
    """纯 Zero-Shot NER+RE 预测器，无 RAG、无检索、无向量索引"""

    def __init__(self, api_key: str, base_url: str, model_name: str):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model_name = model_name

    # ---- 以下方法从 rag_2 复用，逻辑完全一致 ----

    def _normalize_label(self, label: str, label_type: str):
        """归一化标签（实体 or 关系）"""
        if not label:
            return None
        label = label.strip().upper()
        if label_type == "entity":
            if label in VALID_ENTITY_LABELS:
                return label
            for k, (cn, en) in NER_TYPES.items():
                if label in [cn, en, cn.upper(), en.upper(), k]:
                    return k
        elif label_type == "relation":
            if label in VALID_RELATION_LABELS:
                return label
            for k, info in RE_RELATIONS.items():
                if label in [info["name"], info["name"].upper(), k]:
                    return k
        return None

    def _find_text_position(self, text: str, input_text: str, rough_start: int):
        """在原文中基于文本内容查找真实位置"""
        if not text:
            return None
        pos = input_text.find(text)
        if pos != -1:
            return (pos, pos + len(text))
        search_start = max(0, rough_start - 20)
        search_end = min(len(input_text), rough_start + 20 + len(text))
        window = input_text[search_start:search_end]
        pos_in_window = window.find(text)
        if pos_in_window != -1:
            real_pos = search_start + pos_in_window
            return (real_pos, real_pos + len(text))
        return None

    def _validate_entity(self, ent, input_text: str):
        """验证单个实体：基于文本匹配修正位置"""
        if not isinstance(ent, dict):
            return None
        start = ent.get("start")
        end = ent.get("end")
        text = ent.get("text")
        label = ent.get("label")
        if start is None or end is None or text is None or label is None:
            return None
        if not isinstance(text, str) or not text.strip():
            return None
        try:
            start, end = int(start), int(end)
        except (ValueError, TypeError):
            return None
        if not (0 <= start <= len(input_text)):
            return None
        if input_text[start:end] != text:
            corrected = self._find_text_position(text, input_text, start)
            if corrected is None:
                return None
            start, end = corrected
        if not (0 <= start < end <= len(input_text)):
            return None
        if input_text[start:end] != text:
            return None
        norm_label = self._normalize_label(label, "entity")
        if not norm_label:
            return None
        return {"start": start, "end": end, "text": text, "label": norm_label}

    def _validate_relation(self, rel, entities, input_text: str):
        """验证单个关系：基于文本匹配修正 head/tail 位置"""
        if not isinstance(rel, dict):
            return None
        required = ["head", "tail", "head_type", "tail_type", "label"]
        if not all(k in rel for k in required):
            return None
        head_text_raw = rel.get("head", "")
        tail_text_raw = rel.get("tail", "")
        if not head_text_raw or not tail_text_raw:
            return None
        h_pos = input_text.find(head_text_raw)
        t_pos = input_text.find(tail_text_raw)
        if h_pos == -1 or t_pos == -1:
            return None
        h_s, h_e = h_pos, h_pos + len(head_text_raw)
        t_s, t_e = t_pos, t_pos + len(tail_text_raw)
        if not (0 <= h_s < h_e <= len(input_text) and 0 <= t_s < t_e <= len(input_text)):
            return None
        head_type = self._normalize_label(rel["head_type"], "entity")
        tail_type = self._normalize_label(rel["tail_type"], "entity")
        rel_label = self._normalize_label(rel["label"], "relation")
        if not head_type or not tail_type or not rel_label:
            return None
        rel_def = RE_RELATIONS[rel_label]
        if isinstance(rel_def["head"], list) and head_type not in rel_def["head"]:
            return None
        if isinstance(rel_def["tail"], list) and tail_type not in rel_def["tail"]:
            return None
        return {
            "head": head_text_raw,
            "head_start": h_s,
            "head_end": h_e,
            "head_type": head_type,
            "tail": tail_text_raw,
            "tail_start": t_s,
            "tail_end": t_e,
            "tail_type": tail_type,
            "label": rel_label
        }

    def post_process_align(self, result):
        """后处理：对所有预测结果做全局文本对齐校验，剔除仍然错位的实体/关系"""
        input_text = result.get("text", "")
        if not input_text:
            return result

        aligned_entities = []
        seen_keys = set()
        for ent in result.get("entities", []):
            start = ent.get("start")
            end = ent.get("end")
            text = ent.get("text", "")
            label = ent.get("label", "")
            if start is None or end is None or not text:
                continue
            try:
                start, end = int(start), int(end)
            except (ValueError, TypeError):
                continue
            if start < 0 or end > len(input_text) or start > end:
                corrected = self._find_text_position(text, input_text, start)
                if corrected is None:
                    continue
                start, end = corrected
            if input_text[start:end] != text:
                corrected = self._find_text_position(text, input_text, start)
                if corrected is None:
                    continue
                start, end = corrected
            if not (0 <= start < end <= len(input_text)):
                continue
            if input_text[start:end] != text:
                continue
            key = (start, end, label)
            if key in seen_keys:
                continue
            seen_keys.add(key)
            aligned_entities.append({
                "start": start, "end": end, "text": text, "label": label
            })

        aligned_relations = []
        for rel in result.get("relations", []):
            head_text = rel.get("head", "")
            tail_text = rel.get("tail", "")
            if not head_text or not tail_text:
                continue
            h_pos = input_text.find(head_text)
            t_pos = input_text.find(tail_text)
            if h_pos == -1 or t_pos == -1:
                continue
            h_s, h_e = h_pos, h_pos + len(head_text)
            t_s, t_e = t_pos, t_pos + len(tail_text)
            if not (0 <= h_s < h_e <= len(input_text) and 0 <= t_s < t_e <= len(input_text)):
                continue
            aligned_relations.append({
                "head": head_text,
                "head_start": h_s,
                "head_end": h_e,
                "head_type": rel.get("head_type", ""),
                "tail": tail_text,
                "tail_start": t_s,
                "tail_end": t_e,
                "tail_type": rel.get("tail_type", ""),
                "label": rel.get("label", "")
            })

        return {
            "text": input_text,
            "entities": aligned_entities,
            "relations": aligned_relations
        }

    # ---- Zero-Shot 核心方法 ----

    def build_prompt(self, input_text: str) -> str:
        """构建零样本 prompt，包含领域定义、格式要求和一个硬编码 few-shot 示例"""

        entity_desc_lines = []
        entity_desc_lines.append("- CROP: Plant species or genus name, e.g. Tartary buckwheat, sorghum, barley, foxtail millet, oat")
        entity_desc_lines.append("- VAR: Variety or cultivar name / accession number, e.g. JiaYan 2, F-1 hybrids, BaYou 9")
        entity_desc_lines.append("- TRT: Phenotypic trait or characteristic, e.g. drought tolerance, kernel weight, plant height, yield, powdery mildew resistance")
        entity_desc_lines.append("- GST: Developmental or growth stage, e.g. booting stage, seedling stage, flowering stage, grain filling")
        entity_desc_lines.append("- GENE: Gene name or gene family, e.g. ABI5, Dw6, GsNAC2, NAC gene family")
        entity_desc_lines.append("- QTL: Quantitative trait locus / QTL region, e.g. QTL regions, QTLs")
        entity_desc_lines.append("- MRK: Molecular marker (SSR/SNP/InDel etc.), e.g. SNPs, SSR markers")
        entity_desc_lines.append("- CHR: Chromosome or chromosomal region, e.g. chromosome 3B, Chr7")
        entity_desc_lines.append("- BM: Breeding method / experimental technique, e.g. GWAS, CRISPR, positional cloning, targeted deep sequencing, bioinformatics methods")
        entity_desc_lines.append("- CROSS: Parental line or hybrid cross combination")
        entity_desc_lines.append("- ABS: Abiotic stress (drought/salt/heat/cold etc.), e.g. saline-alkali stress, drought stress, 30% field capacity")
        entity_desc_lines.append("- BIS: Biotic stress (pathogen/pest), e.g. E. graminis, powdery mildew")

        relation_desc_lines = []
        relation_desc_lines.append("- CON (包含): Variety belongs to a Crop. Head: VAR, Tail: CROP")
        relation_desc_lines.append("- USE (采用): Variety adopts a Breeding Method. Head: VAR, Tail: BM")
        relation_desc_lines.append("- HAS (具有): Variety possesses a Trait. Head: VAR, Tail: TRT")
        relation_desc_lines.append("- AFF (影响): Abiotic stress / Gene / Marker / QTL affects a Trait. Head: [ABS, GENE, MRK, QTL], Tail: [TRT]")
        relation_desc_lines.append("- OCI (发生于): Trait or stress occurs at a Growth Stage. Head: [TRT, ABS, BIS], Tail: [GST]")
        relation_desc_lines.append("- LOI (定位于): Marker / QTL / Gene located on Chromosome or QTL interval. Head: [MRK, QTL, GENE], Tail: [CHR, QTL]")

        few_shot_input = (
            "The drought-resistant cultivar JiaYan 2 (JIA2) and water-sensitive cultivar BaYou 9 (BA9) "
            "were subjected to three water gradient treatments during the booting stage: "
            "30% field capacity (severe stress), 45% field capacity (moderate stress), and "
            "70% field capacity (normal water supply). After 7 days, root samples were collected "
            "for transcriptome and proteome analyses."
        )

        few_shot_output = json.dumps({
            "text": few_shot_input,
            "entities": [
                {"start": 4, "end": 30, "text": "drought-resistant cultivar", "label": "TRT"},
                {"start": 31, "end": 39, "text": "JiaYan 2", "label": "VAR"},
                {"start": 41, "end": 45, "text": "JIA2", "label": "VAR"},
                {"start": 51, "end": 75, "text": "water-sensitive cultivar", "label": "TRT"},
                {"start": 76, "end": 83, "text": "BaYou 9", "label": "VAR"},
                {"start": 85, "end": 88, "text": "BA9", "label": "VAR"},
                {"start": 151, "end": 164, "text": "booting stage", "label": "GST"},
                {"start": 166, "end": 184, "text": "30% field capacity", "label": "ABS"},
                {"start": 186, "end": 199, "text": "severe stress", "label": "ABS"},
                {"start": 202, "end": 220, "text": "45% field capacity", "label": "ABS"},
                {"start": 222, "end": 237, "text": "moderate stress", "label": "ABS"},
                {"start": 244, "end": 262, "text": "70% field capacity", "label": "ABS"}
            ],
            "relations": [
                {"head": "JiaYan 2", "head_start": 31, "head_end": 39, "head_type": "VAR",
                 "tail": "drought-resistant cultivar", "tail_start": 4, "tail_end": 30, "tail_type": "TRT", "label": "HAS"},
                {"head": "JIA2", "head_start": 41, "head_end": 45, "head_type": "VAR",
                 "tail": "drought-resistant cultivar", "tail_start": 4, "tail_end": 30, "tail_type": "TRT", "label": "HAS"},
                {"head": "BaYou 9", "head_start": 76, "head_end": 83, "head_type": "VAR",
                 "tail": "water-sensitive cultivar", "tail_start": 51, "tail_end": 75, "tail_type": "TRT", "label": "HAS"},
                {"head": "BA9", "head_start": 85, "head_end": 88, "head_type": "VAR",
                 "tail": "water-sensitive cultivar", "tail_start": 51, "tail_end": 75, "tail_type": "TRT", "label": "HAS"},
                {"head": "30% field capacity", "head_start": 166, "head_end": 184, "head_type": "ABS",
                 "tail": "booting stage", "tail_start": 151, "tail_end": 164, "tail_type": "GST", "label": "OCI"},
                {"head": "severe stress", "head_start": 186, "head_end": 199, "head_type": "ABS",
                 "tail": "booting stage", "tail_start": 151, "tail_end": 164, "tail_type": "GST", "label": "OCI"},
                {"head": "45% field capacity", "head_start": 202, "head_end": 220, "head_type": "ABS",
                 "tail": "booting stage", "tail_start": 151, "tail_end": 164, "tail_type": "GST", "label": "OCI"},
                {"head": "moderate stress", "head_start": 222, "head_end": 237, "head_type": "ABS",
                 "tail": "booting stage", "tail_start": 151, "tail_end": 164, "tail_type": "GST", "label": "OCI"},
                {"head": "70% field capacity", "head_start": 244, "head_end": 262, "head_type": "ABS",
                 "tail": "booting stage", "tail_start": 151, "tail_end": 164, "tail_type": "GST", "label": "OCI"}
            ]
        }, ensure_ascii=False, indent=2)

        prompt = f"""You are an expert in agricultural genetics and breeding information extraction. Your task is to identify Named Entities (NER) and Relations (RE) from English scientific text.

=== Entity Types (12 classes) with Definitions and Examples ===
{chr(10).join(entity_desc_lines)}

=== Relation Types (6 classes) with Direction and Constraints ===
{chr(10).join(relation_desc_lines)}

=== Output Format (STRICT JSON) ===
Output a single JSON object with three keys: "text", "entities", and "relations".
- "text": the original input text exactly as provided
- "entities": list of [{{"start": int, "end": int, "text": str, "label": str}}]
- "relations": list of [{{"head": str, "head_start": int, "head_end": int, "head_type": str, "tail": str, "tail_start": int, "tail_end": int, "tail_type": str, "label": str}}]

CRITICAL RULES:
1. All start/end offsets are character-level (0-indexed) and MUST be precise. Verify by slicing the text.
2. The entity "text" field MUST exactly match input_text[start:end]. Double-check character boundaries.
3. Relation head/tail text MUST appear verbatim in the input text. Verify using find().
4. Relation direction (head→tail) MUST follow the constraints defined above.
5. If no entities or relations found, return empty lists [].
6. Do NOT output any text outside the JSON object. No markdown fences. Pure JSON only.

=== Few-Shot Example ===
Input Text: "{few_shot_input}"
Expected Output:
{few_shot_output}

=== Text to Process ===
Input Text: "{input_text}"

Now output the JSON result:"""
        return prompt
    
    

    def call_llm(self, prompt: str, input_text: str, max_retries: int = 3):
        """调用 LLM，带重试 & 安全解析；实体去重复用 rag_2 的 seen_entities set"""
        for attempt in range(1, max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"},
                    temperature=0.0,
                    max_tokens=4096
                )
                content = response.choices[0].message.content

                # 清理可能的 markdown fences
                for delim in ["```json", "```"]:
                    if delim in content:
                        start = content.find(delim) + len(delim)
                        end = content.find("```", start)
                        if end == -1:
                            end = len(content)
                        content = content[start:end].strip()
                        break

                try:
                    raw_result = json.loads(content)
                except json.JSONDecodeError:
                    content = content.strip().rstrip(',').rstrip('}')
                    content = content.replace('\u201c', '"').replace('\u201d', '"')
                    content = content.replace("\u2018", "'").replace("\u2019", "'")
                    try:
                        raw_result = json.loads(content)
                    except Exception:
                        raise ValueError("JSON 解析失败：内容格式严重异常")

                result = {"text": input_text, "entities": [], "relations": []}

                # 实体去重（seen_entities set）
                seen_entities = set()
                for ent in raw_result.get("entities", []):
                    validated = self._validate_entity(ent, input_text)
                    if validated:
                        key = (validated["start"], validated["end"], validated["label"])
                        if key not in seen_entities:
                            seen_entities.add(key)
                            result["entities"].append(validated)

                for rel in raw_result.get("relations", []):
                    validated = self._validate_relation(rel, result["entities"], input_text)
                    if validated:
                        result["relations"].append(validated)

                return result

            except Exception as e:
                print(f"[EXCEPTION] 第 {attempt} 次调用异常: {type(e).__name__}: {e}")
                if attempt < max_retries:
                    time.sleep(2 ** attempt)
                continue

        print("[FATAL] LLM 调用全部失败，返回空结果")
        return {"text": input_text, "entities": [], "relations": []}

    def predict_single(self, input_text: str):
        """零样本预测单条文本：build_prompt + call_llm"""
        if not input_text or not input_text.strip():
            return {"text": input_text, "entities": [], "relations": []}
        prompt = self.build_prompt(input_text)
        return self.call_llm(prompt, input_text=input_text)

关键规则：
所有 start/end 偏移量均为字符级（从 0 开始索引）且必须精确。请通过切片文本来验证。
实体 "text" 字段必须完全匹配 input_text[start:end]。请仔细检查字符边界。
关系 head/tail 文本必须在输入文本中原样出现。请使用 find() 验证。
关系方向 (head->tail) 必须遵循上述定义的约束。
如果未找到实体或关系，返回空列表 []。
不要在 JSON 对象之外输出任何文本。不要使用 Markdown 围栏。仅限纯 JSON。
=== 少样本示例 (Few-Shot Example) ===
输入文本："{few_shot_input}"
预期输出：
{few_shot_output}
=== 待处理文本 ===
输入文本："{input_text}"
现在输出 JSON 结果："""
return prompt

In [3]:
def predict_test_a_file(
    test_file_path: str,
    predictor: ZeroShotNERREPredictor,
    output_path: str = None
) -> list:
    """批量预测 test_A.json（简化版，无 FAISS 依赖检查）"""
    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"测试文件不存在: {test_file_path}")

    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)

    results = []
    total = len(test_data)
    print(f"开始预测 {total} 条记录...")

    for i, item in enumerate(tqdm(test_data, desc="进度")):
        try:
            text = None
            if isinstance(item, dict):
                text = (item.get("text") or item.get("Text")
                        or item.get("sentence") or item.get("content"))
                if not text:
                    keys = [k for k in item.keys()
                            if k.lower() in ['text', 'content', 'abstract', 'paragraph']]
                    if keys:
                        text = item[keys[0]]
            if not text:
                text = str(item)

            if not text.strip():
                result = {"text": "", "entities": [], "relations": [], "error": "空输入"}
            else:
                pred_res = predictor.predict_single(text)
                result = predictor.post_process_align(pred_res)
            results.append(result)

        except Exception as e:
            error_msg = f"第 {i+1} 条处理异常: {type(e).__name__}: {e}"
            print(f"[ERROR] {error_msg}")
            results.append({
                "text": (item.get('text', str(item)) if isinstance(item, dict) else str(item))[:100],
                "entities": [],
                "relations": [],
                "error": str(e)
            })

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"\n✅ 预测完成！结果已保存至: {output_path}")

    return results



# ====================== 主程序入口 ======================
if __name__ == "__main__":
    YOUR_API_KEY = os.getenv("YOUR_MIMO_API_KEY", "sk-cufbww3npqwvfxesehftircxxopf1r9nmmxl9cqlflu6u914")  # 使用 MIMO 对应的 API KEY
    BASE_URL = "https://api.xiaomimimo.com/v1"  # 如 https://api.openai.com/v1 等
    MODEL_NAME = "mimo-v2.5-pro"

    test_file_path = r"C:\Users\28995\Desktop\Heavenly Lake -2\test_A.json"
    output_path = r"E:\codepython\Heavenly Lake_4\submit_zero_shotv2.5pro.json"

    print(f"API Base: {BASE_URL}")
    print(f"Model: {MODEL_NAME}")
    print(f"Input: {test_file_path}")
    print(f"Output: {output_path}")

    try:
        predictor = ZeroShotNERREPredictor(
            api_key=YOUR_API_KEY,
            base_url=BASE_URL,
            model_name=MODEL_NAME
        )
        # 其余代码保持不变...)

        results = predict_test_a_file(
            test_file_path=test_file_path,
            predictor=predictor,
            output_path=output_path
        )

        success_count = sum(1 for r in results if 'error' not in r)
        print(f"\n📊 总计处理 {len(results)} 条，成功 {success_count} 条")

        for i in range(min(2, len(results))):
            r = results[i]
            print(f"\n--- 示例 {i+1} ---")
            txt = r.get("text", "")
            print("原文:", txt[:80] + "..." if len(txt) > 80 else txt)
            print("实体数:", len(r.get("entities", [])))
            print("关系数:", len(r.get("relations", [])))
            if r.get("error"):
                print("⚠️ 错误:", r["error"])

    except Exception as e:
        print(f"❌ 主程序崩溃: {e}")
        import traceback
        traceback.print_exc()

API Base: https://api.xiaomimimo.com/v1
Model: mimo-v2.5-pro
Input: C:\Users\28995\Desktop\Heavenly Lake -2\test_A.json
Output: E:\codepython\Heavenly Lake_4\submit_zero_shotv2.5pro.json
开始预测 400 条记录...


进度:  15%|█▌        | 61/400 [16:59<1:20:41, 14.28s/it]

[EXCEPTION] 第 1 次调用异常: ValueError: JSON 解析失败：内容格式严重异常
[EXCEPTION] 第 2 次调用异常: ValueError: JSON 解析失败：内容格式严重异常


进度:  16%|█▌        | 62/400 [20:37<7:04:42, 75.39s/it]

[EXCEPTION] 第 3 次调用异常: ValueError: JSON 解析失败：内容格式严重异常
[FATAL] LLM 调用全部失败，返回空结果


进度: 100%|██████████| 400/400 [2:04:42<00:00, 18.71s/it]  


✅ 预测完成！结果已保存至: E:\codepython\Heavenly Lake_4\submit_zero_shotv2.5pro.json

📊 总计处理 400 条，成功 400 条

--- 示例 1 ---
原文: The study investigated the molecular mechanism of two Tartary buckwheat genotype...
实体数: 6
关系数: 5

--- 示例 2 ---
原文: Autotetraploid sorghum inbreds have higher kernel weight, seed yield, and protei...
实体数: 11
关系数: 9
